In [ ]:
import os
import json
import shutils
import random
import torch
import pandas as pd
from transformers import MarianMTModel, MarianTokenizer
from tqdm import tqdm
from PIL import Image
from collections import defaultdict
from datasets import load_dataset

import warnings
warnings.filterwarnings('ignore')

/Users/alyani/Desktop/grad school/WOA7015/Final Assessment/FINAL_ASSESSMENT_WOA7015/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# importing slake dataset from Hugging Face

slake_dataset = load_dataset("BoKelvin/SLAKE")


In [3]:
slake_dataset['train'][0]

{'img_name': 'xmlab1/source.jpg',
 'location': 'Abdomen',
 'answer': 'MRI',
 'modality': 'MRI',
 'base_type': 'vqa',
 'answer_type': 'OPEN',
 'question': 'What modality is used to take this image?',
 'qid': 0,
 'content_type': 'Modality',
 'triple': ['vhead', '_', '_'],
 'img_id': 1,
 'q_lang': 'en'}

In [6]:
for data in slake_dataset['train']:
    if data["q_lang"] == "zh":
        zh_example = data
        break

zh_example

{'img_name': 'xmlab0/source.jpg',
 'location': 'Abdomen',
 'answer': 'MRI',
 'modality': 'MRI',
 'base_type': 'vqa',
 'answer_type': 'OPEN',
 'question': '这张图片的成像方式是什么?',
 'qid': 4919,
 'content_type': 'Modality',
 'triple': ['vhead', '_', '_'],
 'img_id': 0,
 'q_lang': 'zh'}

In [10]:
zh_example.keys()

dict_keys(['img_name', 'location', 'answer', 'modality', 'base_type', 'answer_type', 'question', 'qid', 'content_type', 'triple', 'img_id', 'q_lang'])

Notice there is no image file 

In [9]:
# Extract images and organize them in one folder. No duplicates

IMG_DIR = "SLAKE/images"
os.makedirs(IMG_DIR, exist_ok=True)

def save_images(split):
    for sample in slake_dataset[split]:
        img_path = os.path.join(IMG_DIR, sample["img_name"])
        if not os.path.exists(img_path):
            sample["image"].save(img_path)

save_images("train")
save_images("validation")
save_images("test")

KeyError: 'image'

In [ ]:
# language-specific annotations file
# english/chinese split

def extract_annotations(splits):
    en, zh = [], []
    
    for split in dataset[splits]:
        entry = {
            "img_name": split["img_id"],
            "question": split["question"],
            "answer": split["answer"]
        }
        if split["q_lang"] == "en":
            en.append(entry)
        elif split["q_lang"] == "zh":
            zh.append(entry)
    
    return en, zh

train_en, train_zh = extract_annotations("train")
val_en, _ = extract_annotations("Validation")       # ignore val_zh
test_en, _ = extract_annotations("test")            # ignore test_zh

In [ ]:
ANNOT_DIR = "SLAKE/annotations"
os.makedirs(ANNOT_DIR, exists_ok=True)

json.dump(train_en, open(f"{ANNOT_DIR}/train_en.json", "w"), indent=2)
json.dump(val_en, open(f"{ANNOT_DIR}/val_en.json", "w"), indent=2)
json.dump(test_en, open(f"{ANNOT_DIR}/test_en.json", "w"), indent=2)
json.dump(train_zh, open(f"{ANNOT_DIR}/train_zh.json", "w"), indent=2)

In [ ]:
# zh -> en translation pipeline

device = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_NAME = "Helsinki-NLP/opus-mt-zh-en"
tokenizer = MarianTokenizer.from_pretrained(MODEL_NAME)
model = MarianMTModel.from_pretrained(MODEL_NAME).to(device)

def translate_batch(texts, batch_size=8):
    outputs = []
    
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True).to(device)
        with torch.no_grad():
            translated = model.generate(**inputs, max_length=64)
        outputs.extend(tokenizer.batch_decode(translated, skip_special_token=True))
    return outputs

with open("train_zh.json", "r", encoding="utf-8") as f:
    zh_samples = json.load(f)

questions_zh = [sample["question"] for sample in zh_samples]
answers_zh = [sample["answer"] for sample in zh_samples]

questions_zh2en = translate_batch(question_zh)
answers_zh2en = translate_batch(answers_zh)

# saving translated data

translated_samples = []
for sample, q_zh2en, a_zh2en in zip(zh_samples, questions_zh2en, answers_zh2en):
    translated_samples.append({
        "img_name": sample["img_name"],
        "queation": e_zh2en,
        "answer": a_zh2en,
        "source": "zh2en-trnaslated"
    })

with open("train_zh2en.json", "w", encoding="utf-8") as f:
    json.dump(translated_samples, f, indent=2)

print("Tranlsation complete.")

In [ ]:
# merged datasets before training 

def merge_data(en_path, zh2en_path):
    en = json.load(open(en_path))
    zh2en = json.load(open(zh2en_path))
    merged = en + zh2en
    print(f"Merging {len(en)} EN + {len(zh2en)} ZH -> EN = {len(merged)} samples")
    return merged

train_samples = merge_data(
    "SLAKE/annotations/train_en.json",
    "SLAKE/annotations/train_zh2en.json"
)

In [ ]:
# convert dataset to pandas dataframe

train_data = slake_dataset['train'].to_pandas()
validation_data = slake_dataset['validation'].to_pandas()
test_data = slake_dataset['test'].to_pandas()

In [ ]:
train_data.head()

In [ ]:
train_data["answer_type"].value_counts()

In [ ]:
train_data["q_lang"].value_counts()

In [ ]:
# number of image data

len(train_data["img_id"].unique())

In [ ]:
# extract chinese QA to be translated

zh_qa = train_data[train_data["q_lang"] == "zh"]

In [ ]:
zh_qa.head()